# 03 — Pipeline Validation

This notebook validates the complete preprocessing pipeline against the full
selected dataset.

The dataset is processed in chunks to avoid loading the complete dataset into
memory. Statistics are accumulated across all chunks and are used to verify
that the final cleaned dataset satisfies the project requirements.

## Step 1 — Configuration

In [1]:
from pathlib import Path
import json
import pandas as pd

In [2]:
from foresightai.database.connection import get_postgres_connection
from foresightai.database.repositories.review_hash_repository import (
    is_duplicate_hash,
    insert_review_hash,
)
from foresightai.ingestion.preprocessing import (
    preprocess_record,
    preprocess_chunk,
)

In [3]:
print(preprocess_chunk)
print(get_postgres_connection)

<function preprocess_chunk at 0x00000182FF493740>
<function get_postgres_connection at 0x00000182FF158360>


In [4]:
conn = get_postgres_connection()

print("PostgreSQL connection successful.")

conn.close()

PostgreSQL connection successful.


In [5]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

RAW_PATH = Path("../../data/raw/Software.jsonl")

PROCESSED_DIR = Path("../../data/processed")
CLEAN_OUTPUT_PATH = PROCESSED_DIR / "software_clean.jsonl"

REPORT_DIR = Path("../../reports")
REPORT_PATH = REPORT_DIR / "03_pipeline_validation_report.md"


# ============================================================
# PROCESSING CONFIGURATION
# ============================================================

CHUNK_SIZE = 5_000

# Minimum number of clean reviews required by the project
MIN_CLEAN_REVIEWS = 10_000


# ============================================================
# CREATE REQUIRED DIRECTORIES
# ============================================================

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


print("Configuration loaded successfully.")
print(f"Raw dataset      : {RAW_PATH}")
print(f"Chunk size       : {CHUNK_SIZE:,}")
print(f"Minimum reviews  : {MIN_CLEAN_REVIEWS:,}")
print(f"Clean output     : {CLEAN_OUTPUT_PATH}")
print(f"Report output    : {REPORT_PATH}")

Configuration loaded successfully.
Raw dataset      : ..\..\data\raw\Software.jsonl
Chunk size       : 5,000
Minimum reviews  : 10,000
Clean output     : ..\..\data\processed\software_clean.jsonl
Report output    : ..\..\reports\03_pipeline_validation_report.md


## Step 2 - Verifying Input File

In [6]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {RAW_PATH.resolve()}"
    )

print("Raw dataset found.")
print(f"File size: {RAW_PATH.stat().st_size / (1024**3):.2f} GB")

Raw dataset found.
File size: 1.74 GB


In [7]:
if RAW_PATH.suffix.lower() not in {".jsonl", ".json"}:
    raise ValueError(
        f"Expected a JSON/JSONL dataset, got: {RAW_PATH.suffix}"
    )

print(f"Input format: {RAW_PATH.suffix.lower()}")

Input format: .jsonl


## Step 3 — Build the full-dataset chunk reader

In [8]:
def read_dataset_in_chunks(path: Path, chunk_size: int = 5_000):
    """
    Read a JSON/JSONL dataset incrementally using pandas chunks.

    Parameters
    ----------
    path : Path
        Path to the raw dataset.

    chunk_size : int
        Number of records loaded into memory at a time.

    Yields
    ------
    pd.DataFrame
        One chunk of the dataset at a time.
    """

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    suffix = path.suffix.lower()

    if suffix in {".jsonl", ".json"}:

        reader = pd.read_json(
            path,
            lines=True,
            chunksize=chunk_size
        )

        for chunk in reader:
            yield chunk

    elif suffix == ".csv":

        reader = pd.read_csv(
            path,
            chunksize=chunk_size
        )

        for chunk in reader:
            yield chunk

    else:
        raise ValueError(
            f"Unsupported dataset format: {suffix}"
        )

## Step 4 — Create the statistics accumulator

In [9]:
def create_statistics_accumulator():
    """
    Create an empty accumulator for full-dataset preprocessing statistics.
    """

    return {
        "input_records": 0,
        "valid_before_deduplication": 0,
        "duplicate_records": 0,
        "clean_records": 0,
        "chunks_processed": 0,
    }

In [10]:
def update_statistics(total_stats, chunk_stats):
    """
    Add statistics from one processed chunk to the cumulative totals.
    """

    total_stats["chunks_processed"] += 1

    for key in [
        "input_records",
        "valid_before_deduplication",
        "duplicate_records",
        "clean_records",
    ]:
        total_stats[key] += chunk_stats.get(key, 0)

    return total_stats

In [11]:
def print_statistics(stats):
    """
    Display accumulated preprocessing statistics.
    """

    print("=" * 50)
    print("PIPELINE VALIDATION STATISTICS")
    print("=" * 50)

    print(f"Chunks processed              : {stats['chunks_processed']:,}")
    print(f"Input records                 : {stats['input_records']:,}")
    print(
        f"Valid before deduplication    : "
        f"{stats['valid_before_deduplication']:,}"
    )
    print(
        f"Duplicate records             : "
        f"{stats['duplicate_records']:,}"
    )
    print(
        f"Clean records                 : "
        f"{stats['clean_records']:,}"
    )

## Step 5 — Test the reader WITHOUT preprocessing

In [12]:
total_rows_read = 0
chunk_count = 0

for chunk in read_dataset_in_chunks(
    RAW_PATH,
    chunk_size=CHUNK_SIZE
):
    
    chunk_count += 1
    total_rows_read += len(chunk)

    if chunk_count <= 3:
        print(
            f"Chunk {chunk_count}: "
            f"{len(chunk):,} records"
        )

print()
print("=" * 50)
print("READER TEST")
print("=" * 50)
print(f"Chunks read : {chunk_count:,}")
print(f"Rows read   : {total_rows_read:,}")

Chunk 1: 5,000 records
Chunk 2: 5,000 records
Chunk 3: 5,000 records

READER TEST
Chunks read : 977
Rows read   : 4,880,181


## Step 6 — Validate the chunk sizes

In [13]:
chunk_sizes = []

for chunk in read_dataset_in_chunks(
    RAW_PATH,
    chunk_size=CHUNK_SIZE
):
    chunk_sizes.append(len(chunk))

print(f"Number of chunks : {len(chunk_sizes):,}")
print(f"First chunk      : {chunk_sizes[0]:,} records")
print(f"Last chunk       : {chunk_sizes[-1]:,} records")
print(f"Total records    : {sum(chunk_sizes):,}")

Number of chunks : 977
First chunk      : 5,000 records
Last chunk       : 181 records
Total records    : 4,880,181


## Step 7 — Initialize the accumulator

In [14]:
total_stats = create_statistics_accumulator()

print_statistics(total_stats)

PIPELINE VALIDATION STATISTICS
Chunks processed              : 0
Input records                 : 0
Valid before deduplication    : 0
Duplicate records             : 0
Clean records                 : 0


## Step 8 — Process the full dataset

### 8.1 First, create a fresh accumulator

In [15]:
total_stats = create_statistics_accumulator()

### 8.2 Run one validation chunk first

In [16]:
# ============================================================
# SINGLE-CHUNK VALIDATION TEST
# ============================================================

print("Single-chunk validation")
print("=" * 50)

# Create a fresh dataset iterator
iter_full_dataset = read_dataset_in_chunks(
    RAW_PATH,
    chunk_size=CHUNK_SIZE
)

# Get the first chunk
first_chunk = next(iter_full_dataset)

print(f"Input chunk size: {len(first_chunk):,}")

Single-chunk validation
Input chunk size: 5,000


In [17]:
TEXT_COLUMN = "text"
RATING_COLUMN = "rating"

conn = get_postgres_connection()

try:
    clean_chunk, stats = preprocess_chunk(
        chunk=first_chunk,
        text_column=TEXT_COLUMN,
        rating_column=RATING_COLUMN,
        connection=conn
    )

    conn.commit()

finally:
    conn.close()

In [18]:
print("\nSingle-chunk statistics")
print("=" * 50)

for key, value in stats.items():
    print(f"{key}: {value:,}")

print("\nClean records:")
print(clean_chunk.head())


Single-chunk statistics
non_english: 510
empty_or_too_short: 85
duplicate_records: 1
undetectable_language: 2
input_records: 5,000
valid_before_deduplication: 4,403
clean_records: 4,402

Clean records:
   rating                                     title  \
0       5                               Lots of Fun   
1       5                         Light Up The Dark   
2       4                                  Fun game   
3       4  I am not that good at it but my kids are   
4       4                                 good game   

                                                text images        asin  \
0  I love playing tapped out because it is fun to...     []  B00CTQ6SIG   
1  I love this flashlight app!  It really illumin...     []  B0066WJLU6   
2                           One of my favorite games     []  B00KCYMAWK   
3  Cute game. I am not that good at it but my kid...     []  B00P1RK566   
4  Made me think , variety of the puzzles kept it...     []  B00CWY76CC   

  parent_asin  

In [19]:
conn = get_postgres_connection()

try:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE review_dedupe;")

    conn.commit()

finally:
    conn.close()

### 8.3 12K validation

In [20]:
# Truncating database again to ensure a clean state for the next test
conn = get_postgres_connection()

try:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE review_dedupe;")

    conn.commit()

finally:
    conn.close()

In [21]:
# ============================================================
# 12K PIPELINE VALIDATION
# ============================================================

TEST_ROWS = 12_000

total_stats_12k = create_statistics_accumulator()

clean_records_12k = []

rows_processed = 0
chunks_processed = 0

conn = get_postgres_connection()

try:

    for chunk in read_dataset_in_chunks(
        RAW_PATH,
        chunk_size=CHUNK_SIZE
    ):

        # Stop once we have enough records
        remaining = TEST_ROWS - rows_processed

        if remaining <= 0:
            break

        # Take only the required number of records
        chunk = chunk.iloc[:remaining].copy()

        chunks_processed += 1

        print(
            f"Processing chunk {chunks_processed}: "
            f"{len(chunk):,} records"
        )

        # ----------------------------------------------------
        # Preprocess chunk
        # ----------------------------------------------------

        clean_chunk, chunk_stats = preprocess_chunk(
            chunk=chunk,
            text_column=TEXT_COLUMN,
            rating_column=RATING_COLUMN,
            connection=conn
        )

        # ----------------------------------------------------
        # Accumulate statistics
        # ----------------------------------------------------

        update_statistics(
            total_stats_12k,
            chunk_stats
        )

        # ----------------------------------------------------
        # Store clean records
        # ----------------------------------------------------

        if not clean_chunk.empty:
            clean_records_12k.append(clean_chunk)

        rows_processed += len(chunk)

        print(
            f"  Input       : {len(chunk):,}"
        )
        print(
            f"  Valid       : "
            f"{chunk_stats.get('valid_before_deduplication', 0):,}"
        )
        print(
            f"  Duplicates  : "
            f"{chunk_stats.get('duplicate_records', 0):,}"
        )
        print(
            f"  Clean       : "
            f"{chunk_stats.get('clean_records', 0):,}"
        )

        print()

        if rows_processed >= TEST_ROWS:
            break

    conn.commit()

finally:
    conn.close()


# ------------------------------------------------------------
# Combine clean chunks
# ------------------------------------------------------------

if clean_records_12k:
    clean_12k_df = pd.concat(
        clean_records_12k,
        ignore_index=True
    )
else:
    clean_12k_df = pd.DataFrame()


print("=" * 60)
print("12K PIPELINE VALIDATION")
print("=" * 60)

print(f"Input records                 : {total_stats_12k['input_records']:,}")
print(
    f"Valid before deduplication    : "
    f"{total_stats_12k['valid_before_deduplication']:,}"
)
print(
    f"Duplicate records             : "
    f"{total_stats_12k['duplicate_records']:,}"
)
print(
    f"Clean records                 : "
    f"{total_stats_12k['clean_records']:,}"
)
print(f"Chunks processed              : {total_stats_12k['chunks_processed']:,}")
print(f"Clean DataFrame rows          : {len(clean_12k_df):,}")

Processing chunk 1: 5,000 records
  Input       : 5,000
  Valid       : 4,391
  Duplicates  : 1
  Clean       : 4,390

Processing chunk 2: 5,000 records
  Input       : 5,000
  Valid       : 4,365
  Duplicates  : 1
  Clean       : 4,364

Processing chunk 3: 2,000 records
  Input       : 2,000
  Valid       : 1,809
  Duplicates  : 0
  Clean       : 1,809

12K PIPELINE VALIDATION
Input records                 : 12,000
Valid before deduplication    : 10,565
Duplicate records             : 2
Clean records                 : 10,563
Chunks processed              : 3
Clean DataFrame rows          : 10,563


In [22]:
# ============================================================
# 12K CONSISTENCY CHECKS
# ============================================================

assert total_stats_12k["input_records"] == TEST_ROWS, (
    "Input record count does not equal the requested 12K test size."
)

assert (
    total_stats_12k["clean_records"]
    == len(clean_12k_df)
), (
    "Clean-record statistics do not match the clean DataFrame size."
)

assert (
    total_stats_12k["clean_records"]
    <= total_stats_12k["valid_before_deduplication"]
), (
    "Clean records cannot exceed valid records."
)

assert (
    total_stats_12k["valid_before_deduplication"]
    + (
        total_stats_12k["input_records"]
        - total_stats_12k["valid_before_deduplication"]
    )
    == total_stats_12k["input_records"]
)

print("All 12K consistency checks passed.")

All 12K consistency checks passed.


In [23]:
conn = get_postgres_connection()

try:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT COUNT(*) FROM review_dedupe;"
        )

        postgres_hash_count = cur.fetchone()[0]

finally:
    conn.close()

print(
    f"PostgreSQL stored hashes : "
    f"{postgres_hash_count:,}"
)

print(
    f"Clean records             : "
    f"{total_stats_12k['clean_records']:,}"
)

PostgreSQL stored hashes : 10,563
Clean records             : 10,563


In [24]:
clean_12k_df[
    [
        "rating",
        "title",
        "text",
        "text_normalized",
        "title_normalized",
        "review_hash"
    ]
].sample(
    10,
    random_state=42
)

,rating,title,text,text_normalized,title_normalized,review_hash
3282,2,Best thing about this?,It's not completely terrible. That's all I've ...,It's not completely terrible. That's all I've ...,Best thing about this?,ea64c284f875a9b65b1d3280beabd6a9a82a41c1b19fad...
5325,2,$200 for a second-rate OS..?,I bought a $200 copy of Windows XP to run on m...,I bought a $200 copy of Windows XP to run on m...,$200 for a second-rate OS..?,74fb406b3734bb9b1d84213ab5e4d0bc1a2a098bc0e1c5...
4609,1,way too many commercials,way too many commercials,way too many commercials,way too many commercials,f2fbcac07460022deba7907a81a62a7517b0dfae6e39c2...
1562,5,Love this Service,Super Happy with this purchase. It is wonderfu...,Super Happy with this purchase. It is wonderfu...,Love this Service,888638e979e825ffb6f6be4f75657edfa5b600773ee106...
10149,2,Game was great but..,Loved the game would have giving it 5 srars bu...,Loved the game would have giving it 5 srars bu...,Game was great but..,c691c20bfda20455b888c9b42b2c59fe7e929c100b9aff...
2260,4,Fun For Something Different,Fun to play. Seems to hit pretty well. Kind of...,Fun to play. Seems to hit pretty well. Kind of...,Fun For Something Different,ad7549f0f40955df921512784fa3e40bbfbd5432441f46...
6518,1,Mario wont connect,"App used to be Fun for my kids, but in August ...","App used to be Fun for my kids, but in August ...",Mario wont connect,159dc043ababa7c187f12f61a2c8a01ce285de6ee4bd86...
8081,3,Slowest Game Play EVER!,This game would be REALLY FUN if it wasn't so ...,This game would be REALLY FUN if it wasn't so ...,Slowest Game Play EVER!,7a83a348114eba874af39076c76b0bb3598beea0f81129...
8314,3,will not launch when charger is plugged in,App will not launch when put on charge. I ens...,App will not launch when put on charge. I ensu...,will not launch when charger is plugged in,c3ba942c361e3a65ff8f68d5575d2a39c440bf60bd5b92...
9704,3,No worky bb10,As much as I like and use Opera on other devic...,As much as I like and use Opera on other devic...,No worky bb10,4518fe52030a3f0d4014cac4c3bd9dd72398c0f638a5c4...


### 8.4 Full-dataset validation 

In [25]:
# ============================================================
# RESET DEDUPLICATION STATE
# ============================================================

conn = get_postgres_connection()

try:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE review_dedupe;")

    conn.commit()

    print("review_dedupe table truncated successfully.")

finally:
    conn.close()

review_dedupe table truncated successfully.


In [26]:
# ============================================================
# PREPARE CLEAN OUTPUT
# ============================================================

if CLEAN_OUTPUT_PATH.exists():
    CLEAN_OUTPUT_PATH.unlink()
    print("Existing clean output removed.")

print(f"Clean output ready: {CLEAN_OUTPUT_PATH}")
# if output file already exists, remove it to ensure a clean state for the next test

Existing clean output removed.
Clean output ready: ..\..\data\processed\software_clean.jsonl


In [27]:
# ============================================================
# FULL-DATASET PIPELINE VALIDATION
# ============================================================

import time

TEXT_COLUMN = "text"
RATING_COLUMN = "rating"

total_stats = create_statistics_accumulator()

start_time = time.time()

write_header = True

conn = get_postgres_connection()

try:

    for chunk_number, chunk in enumerate(
        read_dataset_in_chunks(
            RAW_PATH,
            chunk_size=CHUNK_SIZE
        ),
        start=1
    ):

        # ----------------------------------------------------
        # Process one chunk
        # ----------------------------------------------------

        clean_chunk, chunk_stats = preprocess_chunk(
            chunk=chunk,
            text_column=TEXT_COLUMN,
            rating_column=RATING_COLUMN,
            connection=conn
        )

        # ----------------------------------------------------
        # Commit PostgreSQL deduplication changes
        # ----------------------------------------------------

        conn.commit()

        # ----------------------------------------------------
        # Update global statistics
        # ----------------------------------------------------

        update_statistics(
            total_stats,
            chunk_stats
        )

        # ----------------------------------------------------
        # Append clean records to JSONL
        # ----------------------------------------------------

        if not clean_chunk.empty:

            clean_chunk.to_json(
                CLEAN_OUTPUT_PATH,
                orient="records",
                lines=True,
                mode="w" if write_header else "a",
                force_ascii=False,
                date_format="iso"
            )

            write_header = False

        # ----------------------------------------------------
        # Progress reporting
        # ----------------------------------------------------

        if chunk_number % 50 == 0 or chunk_number == 1:

            elapsed = time.time() - start_time

            print(
                f"Processed {chunk_number:,} chunks | "
                f"Input records: "
                f"{total_stats['input_records']:,} | "
                f"Clean records: "
                f"{total_stats['clean_records']:,} | "
                f"Elapsed: {elapsed / 60:.1f} min"
            )

finally:

    conn.close()


elapsed = time.time() - start_time

print()
print("=" * 70)
print("FULL-DATASET VALIDATION COMPLETED")
print("=" * 70)

print(
    f"Chunks processed              : "
    f"{total_stats['chunks_processed']:,}"
)

print(
    f"Input records                 : "
    f"{total_stats['input_records']:,}"
)

print(
    f"Valid before deduplication    : "
    f"{total_stats['valid_before_deduplication']:,}"
)

print(
    f"Duplicate records             : "
    f"{total_stats['duplicate_records']:,}"
)

print(
    f"Clean records                 : "
    f"{total_stats['clean_records']:,}"
)

print(
    f"Processing time               : "
    f"{elapsed / 60:.2f} minutes"
)

print(
    f"Clean output                  : "
    f"{CLEAN_OUTPUT_PATH}"
)

Processed 1 chunks | Input records: 5,000 | Clean records: 4,406 | Elapsed: 0.4 min
Processed 50 chunks | Input records: 250,000 | Clean records: 213,724 | Elapsed: 46.1 min


KeyboardInterrupt: 

In [ ]:
# Verify the expected input count
EXPECTED_INPUT_ROWS = 4_880_181

assert total_stats["input_records"] == EXPECTED_INPUT_ROWS, (
    f"Expected {EXPECTED_INPUT_ROWS:,} records, "
    f"but processed {total_stats['input_records']:,}."
)

print("Input record count: PASS")

In [ ]:
# Verify clean record
assert total_stats["clean_records"] > 10_000

print(
    f"Minimum clean-record requirement: PASS "
    f"({total_stats['clean_records']:,} records)"
)

In [ ]:
# Verify basic accounting
assert (
    total_stats["clean_records"]
    <= total_stats["valid_before_deduplication"]
    <= total_stats["input_records"]
)

print("Record accounting: PASS")

In [ ]:
# Verify chunk count
assert total_stats["chunks_processed"] == 977

print("Chunk count: PASS")

In [ ]:
# verify Postgres
conn = get_postgres_connection()

try:

    with conn.cursor() as cur:

        cur.execute(
            "SELECT COUNT(*) FROM review_dedupe;"
        )

        postgres_hash_count = cur.fetchone()[0]

finally:
    conn.close()

print(
    f"PostgreSQL dedupe hashes : "
    f"{postgres_hash_count:,}"
)

print(
    f"Clean records             : "
    f"{total_stats['clean_records']:,}"
)

In [ ]:
# Verify generated output file
print(
    f"Output exists: "
    f"{CLEAN_OUTPUT_PATH.exists()}"
)

print(
    f"Output size: "
    f"{CLEAN_OUTPUT_PATH.stat().st_size / (1024**3):.2f} GB"
)